In [ ]:
# Removendo mensagens de erro:
import warnings

warnings.filterwarnings(
    "ignore",
    message=r".*is_datetime64tz_dtype is deprecated.*",
    category=DeprecationWarning,
)
# Limpando outputs!
from IPython.display import clear_output

## 1. Extraindo os Dados:

#### Importando bibliotecas essenciais para a Análise de Dados:

In [ ]:
# Importando bibliotecas:
from datetime import time, timedelta, datetime              # Corrigindo os horários e as datas...
import pyarrow.parquet as pq                                # PyArrow para trabalhar com Parquet (formato colunar compactado muito usado em data analytics)...
from pathlib import Path                                    # Pegando o endereço do arquivo...
from glob import glob                                       # Busca de arquivos por padrões...
import pyarrow as pa                                        # Apache Arrow, um formato/engine de dados em memória colunar, muito eficiente...
import pandas as pd                                         # Análise e manipulação de dados...
import numpy as np                                          # Tratamento de Dados...
import warnings                                             # Controle de avisos...
import sqlite3                                              # Biblioteca para criar a base de dados em Data Warehouse...
import os                                                   # Operações com arquivos e sistema...
import re                                                   # Mechendo no endereço dos arquivos...import os

pd.set_option('mode.chained_assignment', None)              # Desativa avisos de atribuição encadeada...
warnings.filterwarnings("ignore", category=FutureWarning)   # Oculta avisos de futura depreciação...

#### Consumindo os Dados dos arquivos Parquet:

In [ ]:
# Caminhos das pastas com os arquivos '.parquet':
pasta_ativos = R"..\Base de Dados Original\parquet\Ativos\Ativos.parquet"
pasta_desligados = R"..\Base de Dados Original\parquet\Desligados\Desligados.parquet"
pasta_mensalistas = R"..\Base de Dados Original\parquet\Mensalistas"
pasta_gptw = R"..\Base de Dados Original\parquet\GPTW\GPTW.parquet"
pasta_executivos = R"..\Base de Dados Original\parquet\Executivos"
pasta_horistas = R"..\Base de Dados Original\parquet\Horistas"
pasta_movimentacoes = R"..\Base de Dados Original\parquet\Movimentacoes\Movimentacoes.parquet"

# Lista todos os arquivos '.parquet' em cada pasta:
arquivos_parquet_executivos = glob(os.path.join(pasta_executivos, "*.parquet"))
arquivos_parquet_mensalistas = glob(os.path.join(pasta_mensalistas, "*.parquet"))
arquivos_parquet_horistas = glob(os.path.join(pasta_horistas, "*.parquet"))

# Carrega os arquivos e adiciona a coluna 'Cargo':
tables = [
    pd.read_parquet(caminho).assign(Cargo=cargo)
    for arquivos, cargo in zip([arquivos_parquet_executivos, arquivos_parquet_mensalistas, arquivos_parquet_horistas], ['E', 'M', 'H'])
    for caminho in arquivos
]

# Carrega o arquivo para dentro do Data Frame:
df_desligados = pd.read_parquet(pasta_desligados)
df_ativos = pd.read_parquet(pasta_ativos)
df_gptw = pd.read_parquet(pasta_gptw)
df_movimentacoes = pd.read_parquet(pasta_movimentacoes)

In [ ]:
# === Resumo final ===
total_linhas = sum(len(df) for df in tables)
print("\n📊 RESUMO GERAL")
print(f"Arquivos executivos: {len(arquivos_parquet_executivos)}")
print(f"Arquivos mensalistas: {len(arquivos_parquet_mensalistas)}")
print(f"Arquivos horistas: {len(arquivos_parquet_horistas)}")
print(f"Total de DataFrames lidos: {len(tables)}")
print(f"Total de linhas combinadas: {total_linhas:,}")

## 2. Tratando os Dados:

#### Definição das Funções para Transformação e Tratamento de Dados:
1. **Renomear Colunas:**  
   Função que renomeia as colunas do DataFrame com base em um dicionário de mapeamento, garantindo padronização e clareza nos nomes.
2. **Mover Colunas:**  
   Função que move as colunas do DataFrame com base em um dicionário de mapeamento, garantindo padronização e clareza nos nomes.

In [ ]:
# Função para renomear coluna:
def renomear_coluna(df, col, newname):
    df = df.rename(columns={col: newname})
    return df

# Função para mover coluna
def mover_coluna(df, col, nova_posicao):
    colunas = list(df.columns)
    colunas.insert(nova_posicao, colunas.pop(colunas.index(col)))
    return df[colunas]

#### Aplicando Funções a lista de tabelas:

In [ ]:
# Aplicando a função a todas as colunas de todas as tabelas:
for i in range(len(tables)):
    clear_output(wait=True)
    print(f"Processando tabela {i+1}/{len(tables)}", end="", flush=True)
    tables[i] = renomear_coluna(tables[i], tables[i].columns[0], 'Cadastro')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[1], 'Coletor')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[2], 'Descricao Coletor')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[3], 'Nome')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[4], 'Filial')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[5], 'Data')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[6], 'Hora')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[7], 'Acesso')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[8], 'Dir')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[9], 'Tipo')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[10], 'Uso Acesso')
    tables[i] = renomear_coluna(tables[i], tables[i].columns[11], 'Planta')
#'Cadastro', 'Nome', 'Filial', 'Data', 'Hora', 'Dir', 'Tipo', 'Acesso', 'Coletor', 'Planta', 'Descricao Coletor', 'Uso Acesso', 'Cargo'...

# Aplicando a função de renomear o nome da coluna nas colunas da tabela 'df_ativos':
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[0], 'Cadastro')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[1], 'Cargo')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[2], 'Nome completo')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[3], 'Heranca')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[4], 'Cargo Especifico')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[5], 'Expatriado')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[6], 'Departamento')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[7], 'Área')
df_ativos = renomear_coluna(df_ativos, df_ativos.columns[8], 'Local')

# Aplicando a função de renomear o nome da coluna 'Departamento' da tabela 'df_desligados':
df_desligados = mover_coluna(df_desligados, 'Diretoria RH', 2)
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[0], 'Cadastro')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[2], 'Heranca')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[3], 'Cargo Especifico')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[4], 'Departamento')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[5], 'Área')
df_desligados = renomear_coluna(df_desligados, df_desligados.columns[6], 'Local')

# Aplicando a função de renomear o nome da coluna nas colunas da tabela 'df_gptw':
df_gptw = renomear_coluna(df_gptw, df_gptw.columns[0], 'Área')

# Aplicando a função de renomear o nome da coluna nas colunas da tabela 'df_movimentacoes':
df_movimentacoes = renomear_coluna(df_movimentacoes, df_movimentacoes.columns[0], 'Cadastro')

#### Concatenar todas as tabelas da lista em um único Data Frame:

In [ ]:
# Concatenando todas as tabelas em um único DataFrame:
df_ronda = pd.concat(tables, ignore_index=True)

# Ordenando as colunas na ordem desejada:
df_ronda = df_ronda[['Cadastro', 'Nome', 'Filial', 'Data', 'Hora', 'Dir', 'Tipo', 'Acesso', 'Planta', 'Coletor', 'Descricao Coletor', 'Uso Acesso', 'Cargo']]
df_movimentacoes = df_movimentacoes[
    ['Cadastro', 'Abreviação UO antiga', 'Desc. UO antiga', 'Desc. Cargo antigo',
    'Abreviação UO', 'Desc. UO', 'Desc. Cargo', 'Tipo da Movimentação', 'Data de alteração']
]
df_ronda = df_ronda.drop(df_ronda.index[-1]) # Remover última linha de total...

#### Diferenciando as pessoas que foram demitidas ou que não serão analisadas e inserindo todos os df_ativos delas:

In [ ]:
# Criando toda a função para encontrar as pessoas desligadas não registradas nas bases 'df_ativos' e 'df_desligados':
TARGET_COLS = [
    "Cadastro", "Cargo",
    "Nome completo", "Heranca", "Cargo Especifico",
    "Departamento", "Área", "Local", "Data_Adm", "Data_Dem"
]

def _to_dt(s, dayfirst=True):
    return pd.to_datetime(s, errors="coerce", dayfirst=dayfirst)

def _coalesce(a, b):
    return a.where(~a.isna(), b)

def _fix_cols_ativos(df: pd.DataFrame) -> pd.DataFrame:
    """
    Garante que df_ativos tenha:
      - colunas únicas (sem duplicadas)
      - 'Cargo' e 'Cargo Especifico' separados (se vierem duplicados como 'Cargo Especifico')
    Regra aplicada:
      - Se houver 2+ 'Cargo Especifico' e não existir 'Cargo', renomeia a 1ª ocorrência para 'Cargo'
      - Se houver só 1 'Cargo Especifico' e não existir 'Cargo', cria 'Cargo' copiando 'Cargo Especifico'
    """
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()

    cols = pd.Series(df.columns)
    mask_ce = cols.eq("Cargo Especifico")

    if "Cargo" not in cols.values:
        if mask_ce.sum() >= 2:
            idxs = np.flatnonzero(mask_ce.values)
            cols.iloc[idxs[0]] = "Cargo"  # 1ª ocorrência vira Cargo
        elif mask_ce.sum() == 1:
            # cria Cargo copiando Cargo Especifico (melhor do que NA)
            df["Cargo"] = df["Cargo Especifico"]

    df.columns = cols.tolist()

    # remove duplicadas (se ainda restar alguma)
    df = df.loc[:, ~df.columns.duplicated(keep="last")]
    return df

def _dedup_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()
    return df.loc[:, ~df.columns.duplicated(keep="last")]

def sincronizar_desligamentos(df_movimentacoes, df_ativos, df_desligados, verbose=True):
    mov = _dedup_cols(df_movimentacoes)
    ativos = _fix_cols_ativos(df_ativos)          # <-- AQUI resolve "Cargo Especifico" duplicado
    deslig = _dedup_cols(df_desligados)

    # --- Normaliza Cadastro (Int64 compatível com NA) ---
    for df in (mov, ativos, deslig):
        df["Cadastro"] = pd.to_numeric(df["Cadastro"], errors="coerce").astype("Int64")

    # --- Garante que df_desligados tenha todas as colunas alvo ---
    for c in TARGET_COLS:
        if c not in deslig.columns:
            deslig[c] = pd.NA
    deslig = deslig[TARGET_COLS].copy()

    # --- Parse datas existentes do df_desligados ---
    deslig["Data_Adm_dt"] = _to_dt(deslig["Data_Adm"], dayfirst=True)
    deslig["Data_Dem_dt"] = _to_dt(deslig["Data_Dem"], dayfirst=True)

    # --- Filtra desligamentos nas movimentações ---
    mov["Tipo da Movimentação"] = mov["Tipo da Movimentação"].astype(str).str.strip()
    mov["Data de alteração_dt"] = _to_dt(mov["Data de alteração"], dayfirst=True)

    mov_desl = mov[mov["Tipo da Movimentação"].str.contains("deslig", case=False, na=False)].copy()

    if mov_desl.empty:
        if verbose:
            print("ℹ️ Nenhum 'Desligamento' encontrado em df_movimentacoes.")
        deslig = deslig.drop(columns=["Data_Adm_dt", "Data_Dem_dt"], errors="ignore")
        return ativos, deslig

    cols_mov_util = ["Cadastro", "Data de alteração_dt"]

    if "Cargo" in mov_desl.columns:
        cols_mov_util.append("Cargo")

    for cand in ["Abreviação UO", "Abreviação UO antiga"]:
        if cand in mov_desl.columns:
            cols_mov_util.append(cand); break
    for cand in ["Desc. UO", "Desc. UO antiga"]:
        if cand in mov_desl.columns:
            cols_mov_util.append(cand); break
    for cand in ["Desc. Cargo", "Desc. Cargo antigo"]:
        if cand in mov_desl.columns:
            cols_mov_util.append(cand); break

    mov_last = (
        mov_desl.sort_values("Data de alteração_dt")
               .groupby("Cadastro", as_index=False)
               .tail(1)[cols_mov_util]
               .rename(columns={"Data de alteração_dt": "Data_Dem_dt"})
    )

    incoming = pd.DataFrame({"Cadastro": mov_last["Cadastro"]})
    incoming["Cargo"] = mov_last["Cargo"] if "Cargo" in mov_last.columns else pd.NA

    incoming["Nome completo"] = pd.NA
    incoming["Heranca"] = pd.NA
    incoming["Local"] = pd.NA
    incoming["Data_Adm_dt"] = pd.NaT
    incoming["Data_Dem_dt"] = mov_last["Data_Dem_dt"]

    col_abrev = next((c for c in ["Abreviação UO", "Abreviação UO antiga"] if c in mov_last.columns), None)
    col_uo = next((c for c in ["Desc. UO", "Desc. UO antiga"] if c in mov_last.columns), None)
    col_cargo = next((c for c in ["Desc. Cargo", "Desc. Cargo antigo"] if c in mov_last.columns), None)

    incoming["Área"] = mov_last[col_abrev] if col_abrev else pd.NA
    incoming["Departamento"] = mov_last[col_uo] if col_uo else pd.NA
    incoming["Cargo Especifico"] = mov_last[col_cargo] if col_cargo else pd.NA

    ativos["Data_Admissão_dt"] = _to_dt(ativos.get("Data_Admissão"), dayfirst=True)

    in_ativos = incoming.merge(ativos, on="Cadastro", how="left", suffixes=("", "_a"))

    # (opcional mas recomendado) puxar também o Cargo de ativos, se veio NA em mov
    incoming["Cargo"] = _coalesce(incoming["Cargo"], in_ativos.get("Cargo_a"))

    incoming["Nome completo"] = _coalesce(incoming["Nome completo"], in_ativos.get("Nome completo_a"))
    incoming["Heranca"] = _coalesce(incoming["Heranca"], in_ativos.get("Heranca_a"))
    incoming["Cargo Especifico"] = _coalesce(incoming["Cargo Especifico"], in_ativos.get("Cargo Especifico_a"))
    incoming["Departamento"] = _coalesce(incoming["Departamento"], in_ativos.get("Departamento_a"))
    incoming["Área"] = _coalesce(incoming["Área"], in_ativos.get("Área_a"))
    incoming["Local"] = _coalesce(incoming["Local"], in_ativos.get("Local_a"))
    incoming["Data_Adm_dt"] = _coalesce(incoming["Data_Adm_dt"], in_ativos.get("Data_Admissão_dt"))

    cad_mover = incoming["Cadastro"].dropna().unique()
    ativos_out = ativos[~ativos["Cadastro"].isin(cad_mover)].reset_index(drop=True)

    base = deslig.copy()

    merged = base.merge(
        incoming,
        on="Cadastro",
        how="outer",
        suffixes=("_old", "_new")
    )

    for col in ["Cargo", "Nome completo", "Heranca", "Cargo Especifico", "Departamento", "Área", "Local"]:
        merged[col] = _coalesce(merged.get(f"{col}_old"), merged.get(f"{col}_new"))

    merged["Data_Adm_dt"] = _coalesce(merged.get("Data_Adm_dt_old"), merged.get("Data_Adm_dt_new"))

    old_dem = merged.get("Data_Dem_dt_old")
    new_dem = merged.get("Data_Dem_dt_new")
    merged["Data_Dem_dt"] = old_dem
    merged.loc[merged["Data_Dem_dt"].isna(), "Data_Dem_dt"] = new_dem
    both = old_dem.notna() & new_dem.notna()
    merged.loc[both, "Data_Dem_dt"] = pd.concat([old_dem[both], new_dem[both]], axis=1).max(axis=1)

    merged["Data_Adm"] = pd.to_datetime(merged["Data_Adm_dt"], errors="coerce").dt.strftime("%d.%m.%Y")
    merged["Data_Dem"] = pd.to_datetime(merged["Data_Dem_dt"], errors="coerce").dt.strftime("%d.%m.%Y")

    desligados_out = merged[TARGET_COLS].reset_index(drop=True)

    if verbose:
        print(f"🔎 Desligamentos (movimentações): {len(incoming)}")
        removidos = df_ativos["Cadastro"].isin(cad_mover).sum() if "Cadastro" in df_ativos.columns else 0
        print(f"🧹 Removidos de ativos (se existiam): {removidos}")
        print(f"✅ Ativos: {len(df_ativos)} -> {len(ativos_out)}")
        print(f"✅ Desligados: {len(df_desligados)} -> {len(desligados_out)}")

    ativos_out = ativos_out.drop(columns=["Data_Admissão_dt"], errors="ignore")
    return ativos_out, desligados_out


In [ ]:
# Executando a Função:
df_ativos, df_desligados = sincronizar_desligamentos(
    df_movimentacoes, df_ativos, df_desligados, verbose=True
)


In [ ]:
# Normalização mínima do 'Cadastro' (evita 123, "123", 123.0 divergirem)
def _norm_id(s):
    s = s.astype(str).str.replace(r'\.0$', '', regex=True).str.strip()
    return s.replace({'nan': np.nan})  # volta NaN onde era NaN

for _df in (df_ronda, df_desligados, df_ativos):
    if 'Cadastro' not in _df.columns:
        raise KeyError("Faltou a coluna 'Cadastro' em um dos DataFrames.")
    _df['Cadastro'] = _norm_id(_df['Cadastro'])

# Desligado por Cadastro:
df_ronda['Desligado'] = df_ronda['Cadastro'].isin(df_desligados['Cadastro']).map({True: 'Sim', False: 'Não'})

# Preparar 'df_ativos' com nomes únicos de coluna:
deps = df_ativos[['Cadastro','Nome completo','Área','Departamento','Heranca','Cargo Especifico']].rename(
    columns={
        'Nome completo':'Nome_dep',
        'Área':'Área_dep',
        'Departamento':'Departamento_dep',
        'Heranca':'Heranca_dep',
        'Cargo Especifico':'Cargo Especifico_dep'
    }
)


In [ ]:
# Merge (left) pelo Cadastro:
df_ronda = df_ronda.merge(deps, on='Cadastro', how='left')


In [ ]:
# Dicionários (df_desligados) por Cadastro:
map_df_desligados_nome = dict(zip(df_desligados['Cadastro'], df_desligados['Nome completo']))
map_df_desligados_area = dict(zip(df_desligados['Cadastro'], df_desligados['Área']))
map_df_desligados_dept = dict(zip(df_desligados['Cadastro'], df_desligados['Departamento']))
map_df_desligados_her  = dict(zip(df_desligados['Cadastro'], df_desligados['Heranca']))
map_df_desligados_carg = dict(zip(df_desligados['Cadastro'], df_desligados['Cargo Especifico']))

# Substituir com prioridade: df_desligados > df_ativos > original:

# Nome:
df_ronda['Nome'] = (
    df_ronda['Cadastro'].map(map_df_desligados_nome)   # 1º: df_desligados…
    .fillna(df_ronda['Nome_dep'])                  # 2º: df_ativos…
    .fillna(df_ronda.get('Nome'))                  # 3º: valor original (se existir)…
)

# Área:
df_ronda['Área'] = (
    df_ronda['Cadastro'].map(map_df_desligados_area)
    .fillna(df_ronda['Área_dep'])
)

# Departamento:
df_ronda['Departamento'] = (
    df_ronda['Cadastro'].map(map_df_desligados_dept)
    .fillna(df_ronda['Departamento_dep'])
)

# Heranca:
df_ronda['Heranca'] = (
    df_ronda['Cadastro'].map(map_df_desligados_her)
    .fillna(df_ronda['Heranca_dep'])
)

# Cargo Especifico:
df_ronda['Cargo Especifico'] = (
    df_ronda['Cadastro'].map(map_df_desligados_carg)
    .fillna(df_ronda['Cargo Especifico_dep'])
)

# (Opcional) Manter as colunas *_dep para auditoria; se quiser remover, descomente abaixo:
df_ronda = df_ronda.drop(columns=['Nome_dep','Área_dep','Departamento_dep','Heranca_dep','Cargo Especifico_dep'], errors='ignore')

#### Corrigindo as colunas de Datas:

In [ ]:
# Pareseand por regex:
s = df_ronda["Data"].astype(str).str.strip()

# só datas com barra (ex: 02/09/2026)
mask_slash = s.str.match(r"^\d{1,2}/\d{1,2}/\d{4}$")

# tenta BR primeiro (dd/mm/yyyy) sem warning, porque o format é explícito
br = pd.to_datetime(s.where(mask_slash), format="%d/%m/%Y", errors="coerce")

# onde falhou, tenta US (mm/dd/yyyy) também sem warning
us = pd.to_datetime(s.where(mask_slash), format="%m/%d/%Y", errors="coerce")

# para outros formatos (ex: YYYY-MM-DD), tenta o parser padrão
other = pd.to_datetime(s.where(~mask_slash), errors="coerce")

df_ronda["Data"] = other.fillna(br.fillna(us))


In [ ]:
# Hora (minutos -> time):
df_ronda['Hora'] = pd.to_numeric(df_ronda['Hora'], errors='coerce')
df_ronda['Hora'] = pd.to_timedelta(df_ronda['Hora'], unit='m')
df_ronda['Hora'] = (pd.Timestamp('1900-01-01') + df_ronda['Hora']).dt.time

# Data (MANTER como datetime — não usar strftime aqui):
df_ronda['Data'] = pd.to_datetime(df_ronda['Data'], dayfirst=True, errors='coerce')

# Admissão/Demissão (datetime):
df_desligados['Data_Adm'] = pd.to_datetime(df_desligados['Data_Adm'], dayfirst=True, errors='coerce')
df_desligados['Data_Dem'] = pd.to_datetime(df_desligados['Data_Dem'], dayfirst=True, errors='coerce')
df_ativos['Data_Admissão'] = pd.to_datetime(df_ativos['Data_Admissão'], format='%d.%m.%Y', errors='coerce')

def fmt_dt(x):
    return x.strftime('%d/%m/%Y') if pd.notna(x) else 'NaT'

print(f"df_ronda['Data']           -> min={fmt_dt(df_ronda['Data'].min())} | max={fmt_dt(df_ronda['Data'].max())}")
print(f"df_desligados['Data_Adm']  -> min={fmt_dt(df_desligados['Data_Adm'].min())} | max={fmt_dt(df_desligados['Data_Adm'].max())}")
print(f"df_desligados['Data_Dem']  -> min={fmt_dt(df_desligados['Data_Dem'].min())} | max={fmt_dt(df_desligados['Data_Dem'].max())}")
print(f"df_ativos['Data_Admissão'] -> min={fmt_dt(df_ativos['Data_Admissão'].min())} | max={fmt_dt(df_ativos['Data_Admissão'].max())}")

## 3. Carregando os Dados:

#### Base Inteira para análises específicas:

In [ ]:
# Criando a função para a geração dos arquivos parquet:
def exportar_parquets(dfs: dict[str, pd.DataFrame], base_dir: str = r"..\project\database") -> None:
    """
    Exporta vários DataFrames para .parquet dentro de uma pasta base.
    
    Exemplo de uso:
    exportar_parquets({
        "RegistrosSemanais.parquet": df_registros_semanais,
        "RegistrosDiarios.parquet": df_diarios,
        "Almoco.parquet": df_almoco,
    })
    """
    os.makedirs(base_dir, exist_ok=True)

    for filename, df in dfs.items():
        path = os.path.join(base_dir, filename)
        df.to_parquet(path, index=False)
        print(f"✅Carregado: '{path}'")

### Desenvolvendo toda a Engenharia dos Dados em formato de Data Warehouse:

In [ ]:
# Config
SQLITE_PATH = Path(r"..\project\database\dw_ronda.sqlite")

# =========================
# SQL: STAR SCHEMA (+ STAGING) + VIEWS (APP)
# =========================
DDL_SQL = """
PRAGMA foreign_keys = ON;

-- staging (para carregar em chunks sem explodir memória)
DROP TABLE IF EXISTS stg_evento_acesso;
CREATE TABLE stg_evento_acesso (
  cadastro            TEXT,
  nome                TEXT,
  cargo               TEXT,
  cargo_especifico    TEXT,
  area                TEXT,
  departamento        TEXT,
  heranca             TEXT,
  desligado           TEXT,

  data                TEXT,   -- YYYY-MM-DD (normalizado)
  hora                TEXT,   -- HH:MM:SS (normalizado)
  ts                  TEXT,   -- YYYY-MM-DD HH:MM:SS (normalizado)

  filial              TEXT,
  planta              TEXT,

  coletor             TEXT,
  descricao_coletor   TEXT,

  dir                 TEXT,
  tipo                TEXT,
  acesso              TEXT,
  uso_acesso          TEXT
);

DROP TABLE IF EXISTS fato_evento_acesso;
DROP TABLE IF EXISTS dim_classificacao_acesso;
DROP TABLE IF EXISTS dim_coletor;
DROP TABLE IF EXISTS dim_local;
DROP TABLE IF EXISTS dim_tempo;
DROP TABLE IF EXISTS dim_colaborador;

CREATE TABLE dim_colaborador (
  sk_colaborador      INTEGER PRIMARY KEY AUTOINCREMENT,
  cadastro            TEXT NOT NULL,
  nome                TEXT,
  cargo               TEXT,
  cargo_especifico    TEXT,
  area                TEXT,
  departamento        TEXT,
  heranca             TEXT,
  desligado           TEXT,
  created_at          TEXT DEFAULT (datetime('now')),
  UNIQUE (cadastro)
);

CREATE TABLE dim_tempo (
  sk_tempo        INTEGER PRIMARY KEY AUTOINCREMENT,
  data            TEXT NOT NULL,  -- YYYY-MM-DD
  hora            TEXT NOT NULL,  -- HH:MM:SS
  ano             INTEGER,
  mes             INTEGER,
  dia             INTEGER,
  dia_semana      INTEGER,        -- 0=domingo .. 6=sábado
  hora_num        INTEGER,        -- 0-23
  faixa_horaria   TEXT,
  created_at      TEXT DEFAULT (datetime('now')),
  UNIQUE (data, hora)
);

CREATE TABLE dim_local (
  sk_local    INTEGER PRIMARY KEY AUTOINCREMENT,
  filial      TEXT,
  planta      TEXT,
  created_at  TEXT DEFAULT (datetime('now')),
  UNIQUE (filial, planta)
);

CREATE TABLE dim_coletor (
  sk_coletor         INTEGER PRIMARY KEY AUTOINCREMENT,
  coletor            TEXT,
  descricao_coletor  TEXT,
  created_at         TEXT DEFAULT (datetime('now')),
  UNIQUE (coletor, descricao_coletor)
);

CREATE TABLE dim_classificacao_acesso (
  sk_classificacao_acesso INTEGER PRIMARY KEY AUTOINCREMENT,
  dir                     TEXT,
  tipo                    TEXT,
  acesso                  TEXT,
  uso_acesso              TEXT,
  created_at              TEXT DEFAULT (datetime('now')),
  UNIQUE (dir, tipo, acesso, uso_acesso)
);

CREATE TABLE fato_evento_acesso (
  id_evento                 INTEGER PRIMARY KEY AUTOINCREMENT,
  sk_colaborador             INTEGER NOT NULL,
  sk_tempo                   INTEGER NOT NULL,
  sk_local                   INTEGER NOT NULL,
  sk_coletor                 INTEGER NOT NULL,
  sk_classificacao_acesso    INTEGER NOT NULL,
  qt_evento                  INTEGER NOT NULL DEFAULT 1,
  created_at                 TEXT DEFAULT (datetime('now')),

  FOREIGN KEY (sk_colaborador)          REFERENCES dim_colaborador(sk_colaborador),
  FOREIGN KEY (sk_tempo)                REFERENCES dim_tempo(sk_tempo),
  FOREIGN KEY (sk_local)                REFERENCES dim_local(sk_local),
  FOREIGN KEY (sk_coletor)              REFERENCES dim_coletor(sk_coletor),
  FOREIGN KEY (sk_classificacao_acesso) REFERENCES dim_classificacao_acesso(sk_classificacao_acesso)
);

CREATE INDEX IF NOT EXISTS ix_fato_colaborador ON fato_evento_acesso (sk_colaborador);
CREATE INDEX IF NOT EXISTS ix_fato_tempo       ON fato_evento_acesso (sk_tempo);
CREATE INDEX IF NOT EXISTS ix_fato_local       ON fato_evento_acesso (sk_local);
CREATE INDEX IF NOT EXISTS ix_fato_coletor     ON fato_evento_acesso (sk_coletor);
CREATE INDEX IF NOT EXISTS ix_fato_classif     ON fato_evento_acesso (sk_classificacao_acesso);

DROP VIEW IF EXISTS vw_evento_acesso_flat;

CREATE VIEW vw_evento_acesso_flat AS
SELECT
  f.id_evento,
  c.cadastro, c.nome, c.cargo, c.cargo_especifico, c.area, c.departamento, c.heranca, c.desligado,
  t.data, t.hora, t.ano, t.mes, t.dia, t.dia_semana, t.hora_num, t.faixa_horaria,
  l.filial, l.planta,
  co.coletor, co.descricao_coletor,
  a.dir, a.tipo, a.acesso, a.uso_acesso,
  f.qt_evento
FROM fato_evento_acesso f
JOIN dim_colaborador c           ON c.sk_colaborador = f.sk_colaborador
JOIN dim_tempo t                 ON t.sk_tempo = f.sk_tempo
JOIN dim_local l                 ON l.sk_local = f.sk_local
JOIN dim_coletor co              ON co.sk_coletor = f.sk_coletor
JOIN dim_classificacao_acesso a  ON a.sk_classificacao_acesso = f.sk_classificacao_acesso;

-- ============================================================
-- VIEWS PARA CONSUMO DO APP (contratos "magros")
-- ============================================================

-- (1) Dimensão em PT-BR
DROP VIEW IF EXISTS vw_colaboradores;
CREATE VIEW vw_colaboradores AS
SELECT
  c.cadastro                         AS "Cadastro",
  c.nome                             AS "Nome",
  c.cargo                            AS "Cargo",
  c.cargo_especifico                 AS "Cargo Especifico",
  c.area                             AS "Área",
  c.departamento                     AS "Departamento",
  c.heranca                          AS "Heranca",
  c.desligado                        AS "Desligado"
FROM dim_colaborador c;

-- (2) Eventos válidos para presença (equivalência ETL):
--     Planta != 999 e Tipo != 0
DROP VIEW IF EXISTS vw_evento_acesso_validos_presenca;
CREATE VIEW vw_evento_acesso_validos_presenca AS
SELECT
  cadastro,
  data,
  hora,
  planta,
  tipo
FROM vw_evento_acesso_flat
WHERE
  COALESCE(TRIM(planta), '') <> '' AND TRIM(planta) <> '999'
  AND COALESCE(TRIM(tipo), '') <> '' AND TRIM(tipo) <> '0'
  AND COALESCE(TRIM(data), '') <> '' AND COALESCE(TRIM(hora), '') <> '';

-- (3) Extremos do dia (primeiro e último evento por Cadastro+Data)
DROP VIEW IF EXISTS vw_presenca_extremos_dia;
CREATE VIEW vw_presenca_extremos_dia AS
SELECT
  cadastro,
  data,
  hora,
  planta
FROM (
  SELECT
    cadastro,
    data,
    hora,
    planta,
    MIN(hora) OVER (PARTITION BY cadastro, data) AS hora_min,
    MAX(hora) OVER (PARTITION BY cadastro, data) AS hora_max
  FROM vw_evento_acesso_validos_presenca
)
WHERE hora = hora_min OR hora = hora_max;

-- (4) df_diarios: Cadastro, Planta, Data
DROP VIEW IF EXISTS vw_df_diarios;
CREATE VIEW vw_df_diarios AS
SELECT DISTINCT
  cadastro              AS "Cadastro",
  planta                AS "Planta",
  date(data)            AS "Data"
FROM vw_presenca_extremos_dia;

-- (5) df_registros_semanais: Cadastro, Ano, Semana por Ano, Data, Contagem Presença
-- Semana ISO via âncora na quinta-feira:
--   iso_anchor = date(Data, '-3 days', 'weekday 4')
--   Ano        = strftime('%Y', iso_anchor)
--   Semana     = strftime('%W', iso_anchor) + 1
--   Data       = segunda-feira da semana ISO: date(iso_anchor, '-3 days')
DROP VIEW IF EXISTS vw_df_registros_semanais;
CREATE VIEW vw_df_registros_semanais AS
WITH base AS (
  SELECT
    "Cadastro",
    "Planta",
    "Data" AS data_dia,
    date("Data", '-3 days', 'weekday 4') AS iso_anchor
  FROM vw_df_diarios
)
SELECT
  "Cadastro"                                                AS "Cadastro",
  CAST(strftime('%Y', iso_anchor) AS INTEGER)               AS "Ano",
  CAST(strftime('%W', iso_anchor) AS INTEGER) + 1           AS "Semana por Ano",
  date(iso_anchor, '-3 days')                               AS "Data",
  COUNT(DISTINCT data_dia)                                  AS "Contagem Presença"
FROM base
GROUP BY
  "Cadastro",
  CAST(strftime('%Y', iso_anchor) AS INTEGER),
  CAST(strftime('%W', iso_anchor) AS INTEGER) + 1,
  date(iso_anchor, '-3 days');

-- (6) df_almoco: Cadastro, Data, Hora, Dir, Planta, Coletor, Descricao Coletor
-- Regras ETL:
--   (A) Refeitório: Dir='E' AND coletor contém 'REF' e NÃO contém 'GRILL'/'GRIL'
--   (B) Portaria:  Dir='S' AND coletor contém 'PORT' AND hora entre 11:00 e 14:00
DROP VIEW IF EXISTS vw_df_almoco;
CREATE VIEW vw_df_almoco AS
SELECT
  cadastro                         AS "Cadastro",
  date(data)                       AS "Data",
  hora                             AS "Hora",
  dir                              AS "Dir",
  planta                           AS "Planta",
  coletor                          AS "Coletor",
  descricao_coletor                AS "Descricao Coletor"
FROM vw_evento_acesso_flat
WHERE
  COALESCE(TRIM(data), '') <> '' AND COALESCE(TRIM(hora), '') <> ''
  AND (
    (
      UPPER(COALESCE(dir,'')) = 'E'
      AND UPPER(COALESCE(coletor,'')) LIKE '%REF%'
      AND UPPER(COALESCE(coletor,'')) NOT LIKE '%GRILL%'
      AND UPPER(COALESCE(coletor,'')) NOT LIKE '%GRIL%'
    )
    OR
    (
      UPPER(COALESCE(dir,'')) = 'S'
      AND UPPER(COALESCE(coletor,'')) LIKE '%PORT%'
      AND hora >= '11:00:00' AND hora <= '14:00:00'
    )
  );

-- (7) Placeholders para não quebrar app (substituir quando tiver fato no DW)
DROP VIEW IF EXISTS vw_gptw;
CREATE VIEW vw_gptw AS
SELECT
  CAST(NULL AS TEXT)    AS "Área",
  CAST(NULL AS INTEGER) AS "2025",
  CAST(NULL AS INTEGER) AS "2024",
  CAST(NULL AS INTEGER) AS "2023"
WHERE 0;

DROP VIEW IF EXISTS vw_movimentacoes;
CREATE VIEW vw_movimentacoes AS
SELECT
  CAST(NULL AS TEXT) AS "Cadastro",
  CAST(NULL AS TEXT) AS "Abreviação UO antiga",
  CAST(NULL AS TEXT) AS "Desc. UO antiga",
  CAST(NULL AS TEXT) AS "Desc. Cargo antigo",
  CAST(NULL AS TEXT) AS "Abreviação UO",
  CAST(NULL AS TEXT) AS "Desc. UO",
  CAST(NULL AS TEXT) AS "Desc. Cargo",
  CAST(NULL AS TEXT) AS "Tipo da Movimentação",
  CAST(NULL AS TEXT) AS "Data de alteração"
WHERE 0;

DROP VIEW IF EXISTS vw_headcount;
CREATE VIEW vw_headcount AS
SELECT
  CAST(NULL AS TEXT)    AS "Área",
  CAST(NULL AS INTEGER) AS "2023",
  CAST(NULL AS INTEGER) AS "2024",
  CAST(NULL AS INTEGER) AS "2025"
WHERE 0;
"""

# =========================
# HELPERS
# =========================
def faixa_horaria(h: int) -> str:
    if 5 <= h <= 11:
        return "Manhã"
    if 12 <= h <= 17:
        return "Tarde"
    if 18 <= h <= 23:
        return "Noite"
    return "Madrugada"

def normalize_columns_inplace(df: pd.DataFrame) -> None:
    # NÃO copia o DF (evita MemoryError)
    df.columns = [c.strip() for c in df.columns]

    rename_map = {
        "Cadastro": "cadastro",
        "Nome": "nome",
        "Filial": "filial",
        "Data": "data",
        "Hora": "hora",
        "Dir": "dir",
        "Tipo": "tipo",
        "Acesso": "acesso",
        "Planta": "planta",
        "Coletor": "coletor",
        "Descricao Coletor": "descricao_coletor",
        "Descrição Coletor": "descricao_coletor",
        "Uso Acesso": "uso_acesso",
        "Cargo": "cargo",
        "Desligado": "desligado",
        "Área": "area",
        "Area": "area",
        "Departamento": "departamento",
        "Heranca": "heranca",
        "Herança": "heranca",
        "Cargo Especifico": "cargo_especifico",
        "Cargo Específico": "cargo_especifico",
    }
    df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns}, inplace=True)

# =========================
# PIPELINE (MEMORY-SAFE)
# =========================
def main(df_ronda: pd.DataFrame, chunk_size: int = 200_000):
    # 1) Normaliza nomes de colunas SEM copiar DF inteiro
    normalize_columns_inplace(df_ronda)

    required = [
        "cadastro","nome","filial","data","hora","dir","tipo","acesso","planta",
        "coletor","descricao_coletor","uso_acesso","cargo","desligado","area",
        "departamento","heranca","cargo_especifico"
    ]

    # 2) Cria SQLite e schema
    SQLITE_PATH.parent.mkdir(parents=True, exist_ok=True)
    if SQLITE_PATH.exists():
        SQLITE_PATH.unlink()

    conn = sqlite3.connect(SQLITE_PATH)

    # PRAGMAs de performance (mais rápido, menos durável durante a carga)
    conn.execute("PRAGMA journal_mode = WAL;")       # melhora concorrência e escrita
    conn.execute("PRAGMA synchronous = OFF;")        # mais rápido (risco se faltar energia/crash durante a carga)
    conn.execute("PRAGMA temp_store = MEMORY;")      # temporários em RAM
    conn.execute("PRAGMA cache_size = -200000;")     # ~200MB de cache (valor negativo = KB)
    conn.execute("PRAGMA foreign_keys = ON;")        # mantém sua regra de FK

    try:
        conn.executescript(DDL_SQL)

        # 3) Carrega staging em chunks
        n = len(df_ronda)
        for start in range(0, n, chunk_size):
            end = min(start + chunk_size, n)
            chunk = df_ronda.iloc[start:end].reindex(columns=required, fill_value=pd.NA).copy()

            chunk.replace({"nan": pd.NA, "NaN": pd.NA, "None": pd.NA, "": pd.NA}, inplace=True)

            # data_iso
            data_dt = pd.to_datetime(chunk["data"], errors="coerce")
            data_iso = pd.Series(data_dt.dt.strftime("%Y-%m-%d"), dtype="string")

            # hora_iso
            hora_str = chunk["hora"].astype("string").str.strip()

            # extrai HH:MM ou HH:MM:SS mesmo se vier com sujeira (ex: "15:13:00.....")
            hora_extr = hora_str.str.extract(r"(\d{1,2}:\d{2}(?::\d{2})?)", expand=False)

            # se veio só HH:MM, completa com segundos
            hora_clean = hora_extr.where(hora_extr.str.contains(r":\d{2}:\d{2}$", regex=True), hora_extr + ":00")

            # pad (7:05:00 -> 07:05:00)
            hora_clean = hora_clean.str.replace(r"^(\d):", r"0\1:", regex=True)

            # parse com formato fixo (sem inferência -> sem warning)
            hora_dt = pd.to_datetime(hora_clean, format="%H:%M:%S", errors="coerce")
            hora_iso = pd.Series(hora_dt.dt.strftime("%H:%M:%S"), dtype="string")

            # ts
            ts = pd.to_datetime(
              (data_iso + " " + hora_iso),
              format="%Y-%m-%d %H:%M:%S",
              errors="coerce"
            )
            ts_iso = pd.Series(ts.dt.strftime("%Y-%m-%d %H:%M:%S"), dtype="string")

            stg = pd.DataFrame({
                "cadastro": chunk["cadastro"].astype("string"),
                "nome": chunk["nome"].astype("string"),
                "cargo": chunk["cargo"].astype("string"),
                "cargo_especifico": chunk["cargo_especifico"].astype("string"),
                "area": chunk["area"].astype("string"),
                "departamento": chunk["departamento"].astype("string"),
                "heranca": chunk["heranca"].astype("string"),
                "desligado": chunk["desligado"].astype("string"),
                "data": data_iso,
                "hora": hora_iso,
                "ts": ts_iso,
                "filial": chunk["filial"].astype("string"),
                "planta": chunk["planta"].astype("string"),
                "coletor": chunk["coletor"].astype("string"),
                "descricao_coletor": chunk["descricao_coletor"].astype("string"),
                "dir": chunk["dir"].astype("string"),
                "tipo": chunk["tipo"].astype("string"),
                "acesso": chunk["acesso"].astype("string"),
                "uso_acesso": chunk["uso_acesso"].astype("string"),
            })

            stg.to_sql(
                "stg_evento_acesso",
                conn,
                if_exists="append",
                index=False,
                chunksize=50_000,
                # ALTERAÇÃO: remover method="multi" para não estourar limite de variáveis do SQLite
                method=None
            )

        # índices no staging
        conn.executescript("""
        CREATE INDEX IF NOT EXISTS ix_stg_cadastro  ON stg_evento_acesso (cadastro);
        CREATE INDEX IF NOT EXISTS ix_stg_tempo     ON stg_evento_acesso (data, hora);
        CREATE INDEX IF NOT EXISTS ix_stg_local     ON stg_evento_acesso (filial, planta);
        CREATE INDEX IF NOT EXISTS ix_stg_coletor   ON stg_evento_acesso (coletor, descricao_coletor);
        CREATE INDEX IF NOT EXISTS ix_stg_classif   ON stg_evento_acesso (dir, tipo, acesso, uso_acesso);
        CREATE INDEX IF NOT EXISTS ix_stg_ts        ON stg_evento_acesso (ts);
        """)

        # 4) Dimensões
        conn.executescript("""
        -- ALTERAÇÃO: ts DESC para pegar o mais recente por cadastro (equivalente ao keep='last')
        INSERT INTO dim_colaborador (cadastro, nome, cargo, cargo_especifico, area, departamento, heranca, desligado)
        SELECT cadastro, nome, cargo, cargo_especifico, area, departamento, heranca, desligado
        FROM (
          SELECT
            cadastro, nome, cargo, cargo_especifico, area, departamento, heranca, desligado,
            ROW_NUMBER() OVER (PARTITION BY cadastro ORDER BY ts DESC) AS rn
          FROM stg_evento_acesso
        )
        WHERE rn = 1;

        INSERT INTO dim_tempo (data, hora, ano, mes, dia, dia_semana, hora_num, faixa_horaria)
        SELECT DISTINCT
          s.data AS data,
          s.hora AS hora,
          CAST(strftime('%Y', s.ts) AS INTEGER) AS ano,
          CAST(strftime('%m', s.ts) AS INTEGER) AS mes,
          CAST(strftime('%d', s.ts) AS INTEGER) AS dia,
          CAST(strftime('%w', s.ts) AS INTEGER) AS dia_semana,
          CAST(strftime('%H', s.ts) AS INTEGER) AS hora_num,
          CASE
            WHEN CAST(strftime('%H', s.ts) AS INTEGER) BETWEEN 5 AND 11 THEN 'Manhã'
            WHEN CAST(strftime('%H', s.ts) AS INTEGER) BETWEEN 12 AND 17 THEN 'Tarde'
            WHEN CAST(strftime('%H', s.ts) AS INTEGER) BETWEEN 18 AND 23 THEN 'Noite'
            ELSE 'Madrugada'
          END AS faixa_horaria
        FROM stg_evento_acesso s;

        INSERT INTO dim_local (filial, planta)
        SELECT DISTINCT filial, planta
        FROM stg_evento_acesso;

        INSERT INTO dim_coletor (coletor, descricao_coletor)
        SELECT DISTINCT coletor, descricao_coletor
        FROM stg_evento_acesso;

        INSERT INTO dim_classificacao_acesso (dir, tipo, acesso, uso_acesso)
        SELECT DISTINCT dir, tipo, acesso, uso_acesso
        FROM stg_evento_acesso;
        """)

        # 5) Fato
        conn.execute("BEGIN;")
        conn.executescript("""
        INSERT INTO fato_evento_acesso (
          sk_colaborador, sk_tempo, sk_local, sk_coletor, sk_classificacao_acesso, qt_evento
        )
        SELECT
          c.sk_colaborador,
          t.sk_tempo,
          l.sk_local,
          co.sk_coletor,
          a.sk_classificacao_acesso,
          1 AS qt_evento
        FROM stg_evento_acesso s
        JOIN dim_colaborador c
          ON c.cadastro IS s.cadastro
        JOIN dim_tempo t
          ON t.data IS s.data AND t.hora IS s.hora
        JOIN dim_local l
          ON l.filial IS s.filial AND l.planta IS s.planta
        JOIN dim_coletor co
          ON co.coletor IS s.coletor AND co.descricao_coletor IS s.descricao_coletor
        JOIN dim_classificacao_acesso a
          ON a.dir IS s.dir AND a.tipo IS s.tipo AND a.acesso IS s.acesso AND a.uso_acesso IS s.uso_acesso;
        """)
        conn.commit()

        conn.executescript("DROP TABLE IF EXISTS stg_evento_acesso;")

        print("✅ Arquivo salvo:", SQLITE_PATH)

    finally:
        conn.close()

# Executando
main(df_ronda)
